In [1]:
# Dependency install cell (ultimate one-shot setup for Kani + Qwen HF on SageMaker)
# Run once in a fresh kernel, then restart kernel.

import sys
import subprocess

py = sys.executable

def run(cmd):
  print(">", " ".join(cmd))
  subprocess.check_call(cmd)

# 1) Clean up known conflicting packages
run([py, "-m", "pip", "uninstall", "-y", "torchvision", "transformers", "tokenizers", "kani"])

# 2) Install a compatible set for kani HuggingEngine
run([py, "-m", "pip", "install", "--no-cache-dir", "-U",
   "kani[huggingface]==1.8.0",
   "transformers==4.51.3",
   "tokenizers==0.21.1",
   "accelerate>=0.30.0",
   "huggingface_hub>=0.30.0",
   "sentencepiece",
   "safetensors",
   "protobuf",
   "nest_asyncio",
   "python-dotenv",
   "tqdm"])

# 3) Ensure torch exists (SageMaker usually already has it)
try:
  import torch  # noqa: F401
  print("torch already available")
except Exception:
  run([py, "-m", "pip", "install", "--no-cache-dir", "-U", "torch"])

print("\nInstall complete. IMPORTANT: Restart kernel now, then run Cell 1.")

> /opt/conda/bin/python -m pip uninstall -y torchvision transformers tokenizers kani


Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: tokenizers 0.21.1
Uninstalling tokenizers-0.21.1:
  Successfully uninstalled tokenizers-0.21.1
Found existing installation: kani 1.8.0
Uninstalling kani-1.8.0:
  Successfully uninstalled kani-1.8.0
> /opt/conda/bin/python -m pip install --no-cache-dir -U kani[huggingface]==1.8.0 transformers==4.51.3 tokenizers==0.21.1 accelerate>=0.30.0 huggingface_hub>=0.30.0 sentencepiece safetensors protobuf nest_asyncio python-dotenv tqdm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 106.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 271.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers] [transformers]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-multimodal 1.5.0 requires torchvision<0.25.0,>=0.21.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.


torch already available

Install complete. IMPORTANT: Restart kernel now, then run Cell 1.


In [2]:
# Quick test cell (run after kernel restart)

import importlib.metadata as md

print("python:", __import__("sys").executable)
for p in ["kani", "torch", "transformers", "accelerate", "tokenizers", "huggingface_hub"]:
  try:
      print(f"{p}: {md.version(p)}")
  except Exception:
      print(f"{p}: NOT INSTALLED")

from kani import Kani
from kani.engines.huggingface import HuggingEngine

print("Kani import OK")
print("HuggingEngine import OK")

python: /opt/conda/bin/python
kani: 1.8.0
torch: 2.6.0
transformers: 4.51.3
accelerate: 1.13.0
tokenizers: 0.21.1
huggingface_hub: 0.36.2


2026-03-12 19:24:01.777664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773343441.793364     501 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773343441.798235     501 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-12 19:24:01.814289: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Kani import OK
HuggingEngine import OK


In [3]:
# Cell 1: Setup imports, paths, and Qwen2.5 Kani runtime (DNC, independent)
from pathlib import Path
import os
import re
import json
import time
import asyncio
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from kani import Kani
from kani.engines.huggingface import HuggingEngine

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Put heavy caches on NVMe (recommended in SageMaker)
NVME_CACHE_ROOT = Path("/mnt/sagemaker-nvme/formalizer_cache")
HF_HOME_DIR = NVME_CACHE_ROOT / "hf_home"
HF_HUB_CACHE_DIR = NVME_CACHE_ROOT / "hf_hub"
TRANSFORMERS_CACHE_DIR = NVME_CACHE_ROOT / "transformers"
TORCH_HOME_DIR = NVME_CACHE_ROOT / "torch"
XDG_CACHE_DIR = NVME_CACHE_ROOT / "xdg_cache"

for d in [
  NVME_CACHE_ROOT,
  HF_HOME_DIR,
  HF_HUB_CACHE_DIR,
  TRANSFORMERS_CACHE_DIR,
  TORCH_HOME_DIR,
  XDG_CACHE_DIR,
]:
  d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE_DIR)
os.environ["TORCH_HOME"] = str(TORCH_HOME_DIR)
os.environ["XDG_CACHE_HOME"] = str(XDG_CACHE_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DATA_DIR = PROJECT_ROOT / "data"
DESC_DIR = DATA_DIR / "descriptions"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"

PROMPT_HEADER_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction_dnc_header.txt"
PROMPT_LINE_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction_dnc_line.txt"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_dnc" / "qwen2_5_32b_instruct_kani"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")

# Model setup (same model for LLM-1 header and LLM-2 line generation)
MODEL_NAME = "Qwen2.5-32B-Instruct"
HF_MODEL_ID = f"Qwen/{MODEL_NAME}"

QWEN_ENGINE = HuggingEngine(
  model_id=HF_MODEL_ID,
  model_load_kwargs={
      "device_map": "auto",
      "cache_dir": str(TRANSFORMERS_CACHE_DIR),
  },
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Header prompt exists:", PROMPT_HEADER_PATH.exists())
print("Line prompt exists:", PROMPT_LINE_PATH.exists())
print("Model:", HF_MODEL_ID)
print("NVME cache root:", NVME_CACHE_ROOT)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
No handwritten prompt pipeline was found for the Qwen/Qwen2.5-32B-Instruct model. Kani is falling back to the Hugging Face chat template. For most models this is okay, but you may want to verify that the chat template correctly passes tool calls to the LLM.


PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_dnc/qwen2_5_32b_instruct_kani
VAL exists: True
Domain PDDL exists: True
Header prompt exists: True
Line prompt exists: True
Model: Qwen/Qwen2.5-32B-Instruct
NVME cache root: /mnt/sagemaker-nvme/formalizer_cache


In [4]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 201)]  # p01..p200

PROMPT_HEADER_PREFIX = PROMPT_HEADER_PATH.read_text()
PROMPT_LINE_PREFIX = PROMPT_LINE_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Header prompt chars:", len(PROMPT_HEADER_PREFIX))
print("Line prompt chars:", len(PROMPT_LINE_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 200
First/last: p01 p200
Header prompt chars: 1294
Line prompt chars: 2623
Domain PDDL chars: 997


In [5]:
# Cell 3: Load NL descriptions for one problem ID (original + line-parsed)
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"
  problem_line_path = DESC_DIR / f"{problem_id}_problem_line.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain NL file: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem NL file: {problem_path}")
  if not problem_line_path.exists():
      raise FileNotFoundError(f"Missing line-parsed problem NL file: {problem_line_path}")

  return {
      "domain_nl": domain_path.read_text(),
      "problem_nl": problem_path.read_text(),
      "problem_nl_line": problem_line_path.read_text(),
  }

In [6]:
# Cell 4: Build prompts for one problem (DNC header + DNC line)
def build_header_prompt(problem_id: str):
  parts = load_problem_descriptions(problem_id)
  nl_domain = parts["domain_nl"]
  nl_problem = parts["problem_nl"]

  return (
      f"{PROMPT_HEADER_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Generate ONLY the problem header and object parts "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the problem header and object parts of the problem PDDL for this problem.\n"
      "- Wrap it in exactly one <header>...</header> block.\n"
      "- Do not generate init or goal facts.\n"
      "- Follow the exact formatting constraints in the instruction.\n"
      "- Do not include any extra text before or after <header>...</header>."
  )


def build_line_prompt(problem_id: str, section: str, line_text: str):
  parts = load_problem_descriptions(problem_id)
  nl_domain = parts["domain_nl"]

  return (
      f"{PROMPT_LINE_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Read ONLY ONE LINE of the natural language problem description at a time "
      "and generate ONLY its corresponding PDDL fact line.\n\n"
      "You are given four inputs:\n"
      "1) Original natural language domain description\n"
      "2) Ground-truth domain PDDL (fixed reference)\n"
      "3) Current section label\n"
      "4) Current single natural language line to convert\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "### Section\n"
      f"{section}\n\n"
      "### Current line\n"
      f"{line_text}\n\n"
      "Output contract:\n"
      "- Generate ONLY one corresponding PDDL fact line.\n"
      "- Wrap it in exactly one <fact>...</fact> block.\n"
      "- Do not include any extra explanation or text before or after <fact>...</fact>."
  )

In [7]:
# Cell 5: Call Qwen2.5 via Kani for one prompt
async def _call_qwen_async(prompt: str, temperature: float = 0.4, max_new_tokens: int = 32768):
  ai = Kani(QWEN_ENGINE, system_prompt='')
  return await ai.chat_round_str(
      prompt,
      temperature=temperature,
      max_new_tokens=max_new_tokens,
  )


def call_qwen(prompt: str, temperature: float = 0.4, max_output_tokens: int = 32768):
  try:
      loop = asyncio.get_running_loop()
  except RuntimeError:
      return asyncio.run(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))

  import nest_asyncio
  nest_asyncio.apply()
  return loop.run_until_complete(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))

In [8]:
# Cell 6: Parse model output blocks for DNC (<header> and <fact>)
HEADER_RE = re.compile(r"<header>\s*(.*?)\s*</header>", re.IGNORECASE | re.DOTALL)
FACT_RE = re.compile(r"<fact>\s*(.*?)\s*</fact>", re.IGNORECASE | re.DOTALL)

def extract_header_block(raw_text: str):
  m = HEADER_RE.search(raw_text or "")
  return m.group(1).strip() if m else ""

def extract_fact_block(raw_text: str):
  m = FACT_RE.search(raw_text or "")
  return m.group(1).strip() if m else ""

In [9]:
# Cell 7: Online solver (public endpoint) + raw API JSON persistence
import requests
import re
import json

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()
  res = None

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Persist full final API response for debugging/truncation analysis
  (work_dir / "solver_raw_response.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic for zero-cost empty-plan cases
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [10]:
# Cell 8: Validate plan with VAL (terminal-equivalent capture, no timeout)
import subprocess
import shlex
import re

def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_file_path: Path):
  cmd = f"{shlex.quote(str(VAL_PATH))} {shlex.quote(str(domain_pddl_path))} {shlex.quote(str(problem_pddl_path))} {shlex.quote(str(plan_file_path))}"
  proc = subprocess.run(
      cmd,
      shell=True,
      capture_output=True,
      text=True,
  )

  val_output = (proc.stdout or "") + ("\n" + proc.stderr if proc.stderr else "")

  ok_patterns = [
      r"Plan valid",
      r"Successful plans:",
  ]
  bad_patterns = [
      r"Plan failed to execute",
      r"Bad plan description",
      r"Syntax error",
      r"Error",
      r"Failed",
      r"Unspecified precondition",
      r"goal not satisfied",
  ]

  val_ok = any(re.search(p, val_output, re.IGNORECASE) for p in ok_patterns) and not any(
      re.search(p, val_output, re.IGNORECASE) for p in bad_patterns
  )

  return {
      "val_ok": bool(val_ok),
      "val_output": val_output,
      "returncode": proc.returncode,
  }

In [11]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_header_text: str, final_problem_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output (header model output persisted at top level for consistency)
  (cand_dir / "raw.txt").write_text(raw_header_text or "")

  # Final generated problem PDDL
  (cand_dir / "problem.pddl").write_text(final_problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  return cand_dir

In [12]:
# Cell 10: DNC helpers (split sections, clean facts, compose final PDDL)
def _split_line_problem_sections(problem_line_text: str):
  lines = [line.strip() for line in (problem_line_text or "").splitlines() if line.strip()]

  init_header = "As initial conditions I have that"
  goal_header = "My goal is to have that"

  init_lines = []
  goal_lines = []
  section = None

  for line in lines:
      if line == init_header:
          section = "init"
          continue
      if line == goal_header:
          section = "goal"
          continue

      if section == "init":
          init_lines.append(line)
      elif section == "goal":
          goal_lines.append(line)

  if not init_lines or not goal_lines:
      raise ValueError("Failed to split line-parsed problem into init/goal sections.")
  return init_lines, goal_lines


def _clean_fact_text(fact_text: str):
  fact = (fact_text or "").strip()
  if "\n" in fact:
      fact = fact.splitlines()[0].strip()
  if fact.endswith("."):
      fact = fact[:-1].strip()
  if not fact:
      return ""
  if not (fact.startswith("(") and fact.endswith(")")):
      return ""
  return fact


def _compose_problem_pddl_from_dnc(header_text: str, init_facts: list[str], goal_facts: list[str]):
  header = (header_text or "").strip()

  # Defensive trim if model leaked these sections.
  for leak in ["(:init", "(:goal"]:
      idx = header.find(leak)
      if idx != -1:
          header = header[:idx].rstrip()

  if not header.startswith("(define"):
      raise ValueError("Header block does not start with '(define'.")

  # Remove one trailing close if present so we can append sections.
  if header.endswith(")"):
      header = header.rsplit(")", 1)[0].rstrip()

  lines = [header]
  lines.append("  (:init")
  for f in init_facts:
      lines.append(f"    {f}")
  lines.append("  )")
  lines.append("  (:goal")
  lines.append("    (and")
  for f in goal_facts:
      lines.append(f"      {f}")
  lines.append("    )")
  lines.append("  )")
  lines.append(")")
  return "\n".join(lines) + "\n"

In [13]:
# Cell 11: Run one problem through full DNC pipeline
def process_one_problem(problem_id: str):
  parts = load_problem_descriptions(problem_id)

  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  dnc_dir = cand_dir / "dnc"
  init_dir = dnc_dir / "init_lines"
  goal_dir = dnc_dir / "goal_lines"
  failed_dir = dnc_dir / "failed_lines"
  for d in [cand_dir, dnc_dir, init_dir, goal_dir, failed_dir]:
      d.mkdir(parents=True, exist_ok=True)

  failed_index_path = dnc_dir / "failed_lines_index.json"

  # clean previous failed-lines artifacts for a fresh run
  for p in failed_dir.glob("*"):
      if p.is_file():
          p.unlink()
  if failed_index_path.exists():
      failed_index_path.unlink()

  # 1) Header generation
  header_prompt = build_header_prompt(problem_id)
  raw_header = call_qwen(header_prompt)
  header_text = extract_header_block(raw_header)

  (dnc_dir / "header_prompt.txt").write_text(header_prompt or "")
  (dnc_dir / "header_raw.txt").write_text(raw_header or "")
  (dnc_dir / "header_extracted.txt").write_text(header_text or "")

  if not header_text.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <header>...</header> found in header model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_header, "", solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # 2) Line-by-line fact generation
  init_lines, goal_lines = _split_line_problem_sections(parts["problem_nl_line"])
  init_facts = []
  goal_facts = []
  failed_lines = []

  for i, line in enumerate(init_lines, start=1):
      prompt = build_line_prompt(problem_id, "init", line)
      raw = call_qwen(prompt)
      fact = _clean_fact_text(extract_fact_block(raw))
      init_facts.append(fact)

      line_file = init_dir / f"line_{i:03d}.txt"
      line_file.write_text(
          f"[section]\ninit\n\n"
          f"[line_index]\n{i}\n\n"
          f"[nl_line]\n{line}\n\n"
          f"[prompt]\n{prompt}\n\n"
          f"[raw]\n{raw}\n\n"
          f"[fact]\n{fact}\n"
      )

      if not fact:
          failed_lines.append({"section": "init", "line_index": i, "nl_line": line})
          (failed_dir / f"init_line_{i:03d}.txt").write_text(
              f"[section]\ninit\n\n"
              f"[line_index]\n{i}\n\n"
              f"[nl_line]\n{line}\n\n"
              f"[raw]\n{raw}\n"
          )

  for i, line in enumerate(goal_lines, start=1):
      prompt = build_line_prompt(problem_id, "goal", line)
      raw = call_qwen(prompt)
      fact = _clean_fact_text(extract_fact_block(raw))
      goal_facts.append(fact)

      line_file = goal_dir / f"line_{i:03d}.txt"
      line_file.write_text(
          f"[section]\ngoal\n\n"
          f"[line_index]\n{i}\n\n"
          f"[nl_line]\n{line}\n\n"
          f"[prompt]\n{prompt}\n\n"
          f"[raw]\n{raw}\n\n"
          f"[fact]\n{fact}\n"
      )

      if not fact:
          failed_lines.append({"section": "goal", "line_index": i, "nl_line": line})
          (failed_dir / f"goal_line_{i:03d}.txt").write_text(
              f"[section]\ngoal\n\n"
              f"[line_index]\n{i}\n\n"
              f"[nl_line]\n{line}\n\n"
              f"[raw]\n{raw}\n"
          )

  # persist failed-lines index if any; otherwise keep failed_lines folder empty
  if failed_lines:
      failed_index_path.write_text(json.dumps(failed_lines, indent=2))
  else:
      for p in failed_dir.glob("*"):
          if p.is_file():
              p.unlink()
      if failed_index_path.exists():
          failed_index_path.unlink()

  # keep only valid facts for composition in this run
  init_facts_valid = [f for f in init_facts if f]
  goal_facts_valid = [f for f in goal_facts if f]

  (dnc_dir / "init_facts_combined.txt").write_text(
      ("\n".join(init_facts_valid) + "\n") if init_facts_valid else ""
  )
  (dnc_dir / "goal_facts_combined.txt").write_text(
      ("\n".join(goal_facts_valid) + "\n") if goal_facts_valid else ""
  )

  # 3) Compose final problem.pddl from current successful facts
  final_problem_pddl = _compose_problem_pddl_from_dnc(header_text, init_facts_valid, goal_facts_valid)
  (cand_dir / "problem.pddl").write_text(final_problem_pddl)

  # 4) Solve + VAL
  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=cand_dir / "problem.pddl",
      work_dir=cand_dir,
      timeout_sec=300,
  )

  val_result = None
  if solver_result.get("plan_found"):
      gt_problem_path = DATA_DIR / "pddl" / f"{problem_id}.pddl"
      if gt_problem_path.exists():
          plan_path = cand_dir / "plan.txt"
          plan_path.write_text(solver_result.get("plan_text", "") or "")
          val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)

  save_problem_outputs(problem_id, raw_header, final_problem_pddl, solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result.get("plan_found")),
      "val_ok": bool(val_result.get("val_ok")) if val_result is not None else False,
      "solver_error": solver_result.get("solver_error", ""),
      "failed_line_count": len(failed_lines),
  }

In [14]:
# Cell 12: Helper - re-run selected failed lines and refresh failed_lines state
def rerun_failed_lines(problem_id: str, targets=None):
  """
  targets:
    - None: re-run all currently failed lines
    - int: re-run one failed line by its 1-based position in failed_lines_index.json
    - list[int]: re-run a subset by 1-based positions
  """
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  dnc_dir = cand_dir / "dnc"
  init_dir = dnc_dir / "init_lines"
  goal_dir = dnc_dir / "goal_lines"
  failed_dir = dnc_dir / "failed_lines"
  failed_index_path = dnc_dir / "failed_lines_index.json"

  if not failed_index_path.exists():
      print(f"{problem_id}: no failed lines to re-run.")
      return {"problem": problem_id, "rerun_attempted": 0, "still_failed": 0}

  parts = load_problem_descriptions(problem_id)
  init_lines, goal_lines = _split_line_problem_sections(parts["problem_nl_line"])

  prev_failed = json.loads(failed_index_path.read_text())

  # choose subset to rerun
  if targets is None:
      selected_idx = set(range(1, len(prev_failed) + 1))
  elif isinstance(targets, int):
      selected_idx = {targets}
  else:
      selected_idx = set(int(x) for x in targets)

  selected_failed = []
  untouched_failed = []
  for i, item in enumerate(prev_failed, start=1):
      if i in selected_idx:
          selected_failed.append(item)
      else:
          untouched_failed.append(item)

  newly_still_failed = []
  newly_succeeded = []

  for item in selected_failed:
      section = item["section"]
      line_index = int(item["line_index"])
      nl_line = item["nl_line"]

      prompt = build_line_prompt(problem_id, section, nl_line)
      raw = call_qwen(prompt)
      fact = _clean_fact_text(extract_fact_block(raw))

      if section == "init":
          line_file = init_dir / f"line_{line_index:03d}.txt"
      else:
          line_file = goal_dir / f"line_{line_index:03d}.txt"

      line_file.write_text(
          f"[section]\n{section}\n\n"
          f"[line_index]\n{line_index}\n\n"
          f"[nl_line]\n{nl_line}\n\n"
          f"[prompt]\n{prompt}\n\n"
          f"[raw]\n{raw}\n\n"
          f"[fact]\n{fact}\n"
      )

      if fact:
          newly_succeeded.append({"section": section, "line_index": line_index, "nl_line": nl_line})
      else:
          newly_still_failed.append({"section": section, "line_index": line_index, "nl_line": nl_line})
          (failed_dir / f"{section}_line_{line_index:03d}.txt").write_text(
              f"[section]\n{section}\n\n"
              f"[line_index]\n{line_index}\n\n"
              f"[nl_line]\n{nl_line}\n\n"
              f"[raw]\n{raw}\n"
          )

  # latest failed set = untouched previous + still failed from this rerun
  updated_failed = untouched_failed + newly_still_failed

  # rebuild failed_lines folder
  if failed_dir.exists():
      for p in failed_dir.glob("*"):
          if p.is_file():
              p.unlink()

  if updated_failed:
      failed_dir.mkdir(parents=True, exist_ok=True)
      for item in updated_failed:
          section = item["section"]
          line_index = int(item["line_index"])
          nl_line = item["nl_line"]
          (failed_dir / f"{section}_line_{line_index:03d}.txt").write_text(
              f"[section]\n{section}\n\n"
              f"[line_index]\n{line_index}\n\n"
              f"[nl_line]\n{nl_line}\n"
          )
      failed_index_path.write_text(json.dumps(updated_failed, indent=2))
  else:
      if failed_index_path.exists():
          failed_index_path.unlink()

  # rebuild combined fact files from latest per-line files
  init_facts = []
  for i, line in enumerate(init_lines, start=1):
      line_file = init_dir / f"line_{i:03d}.txt"
      fact = ""
      if line_file.exists():
          txt = line_file.read_text()
          m = re.search(r"\[fact\]\n(.*)", txt, flags=re.DOTALL)
          fact = (m.group(1).strip() if m else "").splitlines()[0].strip() if txt else ""
      if fact:
          init_facts.append(fact)

  goal_facts = []
  for i, line in enumerate(goal_lines, start=1):
      line_file = goal_dir / f"line_{i:03d}.txt"
      fact = ""
      if line_file.exists():
          txt = line_file.read_text()
          m = re.search(r"\[fact\]\n(.*)", txt, flags=re.DOTALL)
          fact = (m.group(1).strip() if m else "").splitlines()[0].strip() if txt else ""
      if fact:
          goal_facts.append(fact)

  (dnc_dir / "init_facts_combined.txt").write_text(("\n".join(init_facts) + "\n") if init_facts else "")
  (dnc_dir / "goal_facts_combined.txt").write_text(("\n".join(goal_facts) + "\n") if goal_facts else "")

  # regenerate final problem.pddl from latest header + latest valid facts
  header_text = (dnc_dir / "header_extracted.txt").read_text() if (dnc_dir / "header_extracted.txt").exists() else ""
  final_problem_pddl = _compose_problem_pddl_from_dnc(header_text, init_facts, goal_facts)
  (cand_dir / "problem.pddl").write_text(final_problem_pddl)

  print(f"{problem_id}: rerun attempted={len(selected_failed)}, succeeded={len(newly_succeeded)}, still_failed={len(updated_failed)}")
  return {
      "problem": problem_id,
      "rerun_attempted": len(selected_failed),
      "rerun_succeeded": len(newly_succeeded),
      "still_failed": len(updated_failed),
  }

In [15]:
# Cell 13: Generate only (DNC, no solver/VAL)
smoke_problem = "p51"

# Generation-only path (mirrors prior notebook style): run DNC generation logic only
result = process_one_problem(smoke_problem)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
problem_pddl_text = (cand_dir / "problem.pddl").read_text() if (cand_dir / "problem.pddl").exists() else ""

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_problem_pddl_chars:", len(problem_pddl_text or ""))
print("parse_ok:", result.get("parse_ok"))
print("plan_found:", result.get("plan_found"))
print("val_ok:", result.get("val_ok"))
print("failed_line_count:", result.get("failed_line_count", 0))
print("saved:", cand_dir)

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Cell 14: Re-run failed lines for one problem (manual single-pass)
rerun_problem = "p01"

# Choose one mode:
# 1) re-run all failed lines:
RERUN_TARGETS = None

# 2) re-run one failed line by 1-based position in failed_lines_index.json:
# RERUN_TARGETS = 1

# 3) re-run selected failed lines by 1-based positions:
# RERUN_TARGETS = [1, 3, 5]

rerun_result = rerun_failed_lines(rerun_problem, targets=RERUN_TARGETS)

cand_dir = OUTPUT_ROOT / rerun_problem / "cand_00"
dnc_dir = cand_dir / "dnc"
failed_index_path = dnc_dir / "failed_lines_index.json"

print("Re-run done.")
print("problem:", rerun_problem)
print("rerun_attempted:", rerun_result.get("rerun_attempted"))
print("rerun_succeeded:", rerun_result.get("rerun_succeeded"))
print("still_failed:", rerun_result.get("still_failed"))
print("updated_problem_pddl:", cand_dir / "problem.pddl")
print("failed_lines_index_exists:", failed_index_path.exists())

In [ ]:
# Cell 15: Solver+VAL only (single problem, no generation)
import contextlib
import io

smoke_problem = "p01"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

problem_path = cand_dir / "problem.pddl"  # final DNC-composed problem for solver
if not problem_path.exists() or not problem_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty generated problem file: {problem_path}")

gt_problem_path = DATA_DIR / "pddl" / f"{smoke_problem}.pddl"  # ground-truth problem for VAL
if not gt_problem_path.exists():
  raise FileNotFoundError(f"Missing ground-truth problem file for VAL: {gt_problem_path}")

# Suppress verbose live logs from solver in notebook output
with contextlib.redirect_stdout(io.StringIO()):
  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=problem_path,
      work_dir=cand_dir,
      timeout_sec=300,
  )

# Save solver logs/results
(cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
(cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

produced_files = ["solver_stdout.txt", "solver_stderr.txt"]
val_result = None

if solver_result["plan_found"]:
  plan_file = cand_dir / "plan.txt"
  plan_text = solver_result.get("plan_text", "")
  plan_file.write_text(plan_text)
  produced_files.append("plan.txt")

  with contextlib.redirect_stdout(io.StringIO()):
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_file)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
  produced_files.append("validator.log")
else:
  (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
  produced_files.append("solver_error.txt")

summary = {
  "problem": smoke_problem,
  "plan_found": bool(solver_result["plan_found"]),
  "val_ok": bool(val_result["val_ok"]) if val_result is not None else False,
  "solver_error": solver_result.get("solver_error", ""),
}

print("Produced files:")
for name in produced_files:
  print("-", name)

if val_result is not None:
  print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
else:
  print("VAL result: not run (solver failed)")

print("Summary:")
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 16: Batch run (sequential, one-by-one)
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 11)]  # change this list/range as needed

batch_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="DNC batch", unit="problem"):
  try:
      res = process_one_problem(pid)
      batch_results.append(res)
      print(
          f"{pid}: parse_ok={res.get('parse_ok')} "
          f"plan_found={res.get('plan_found')} "
          f"val_ok={res.get('val_ok')} "
          f"failed_line_count={res.get('failed_line_count', 0)}"
      )
  except Exception as e:
      batch_results.append(
          {
              "problem": pid,
              "parse_ok": False,
              "plan_found": False,
              "val_ok": False,
              "failed_line_count": -1,
              "solver_error": str(e),
          }
      )
      print(f"{pid}: error -> {e}")

print("\nBatch done.")
print("total:", len(batch_results))
print("parse_ok:", sum(1 for r in batch_results if r.get("parse_ok")))
print("plan_found:", sum(1 for r in batch_results if r.get("plan_found")))
print("val_ok:", sum(1 for r in batch_results if r.get("val_ok")))
print("with_failed_lines:", sum(1 for r in batch_results if r.get("failed_line_count", 0) > 0))

In [ ]:
# Cell 17: In-notebook metrics summary (no persistence)
import pandas as pd

# Define any batch you want here
SUMMARY_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 11)]

rows = []
for pid in SUMMARY_PROBLEMS:
  cand_dir = OUTPUT_ROOT / pid / "cand_00"
  dnc_dir = cand_dir / "dnc"

  solver_stdout_path = cand_dir / "solver_stdout.txt"
  solver_stderr_path = cand_dir / "solver_stderr.txt"
  solver_error_path = cand_dir / "solver_error.txt"
  plan_txt_path = cand_dir / "plan.txt"
  plan_path = cand_dir / "plan"
  val_log_path = cand_dir / "validator.log"
  failed_index_path = dnc_dir / "failed_lines_index.json"

  solver_stdout = solver_stdout_path.read_text() if solver_stdout_path.exists() else ""
  solver_stderr = solver_stderr_path.read_text() if solver_stderr_path.exists() else ""
  solver_error = solver_error_path.read_text().strip() if solver_error_path.exists() else ""

  has_plan_txt = plan_txt_path.exists() and plan_txt_path.read_text().strip() != ""
  has_plan = plan_path.exists() and plan_path.read_text().strip() != ""
  plan_found = bool(has_plan_txt or has_plan)

  val_log = val_log_path.read_text() if val_log_path.exists() else ""
  val_ran = val_log_path.exists() and val_log.strip() != ""
  val_ok = "Plan valid" in val_log

  if failed_index_path.exists():
      try:
          failed_line_count = len(json.loads(failed_index_path.read_text()))
      except Exception:
          failed_line_count = -1
  else:
      failed_line_count = 0

  if val_ok:
      final_status = "success"
  elif plan_found and not val_ok:
      final_status = "plan_invalid"
  elif (not plan_found) and solver_error:
      final_status = "solver_failed"
  else:
      final_status = "other"

  rows.append(
      {
          "problem": pid,
          "final_status": final_status,
          "plan_found": plan_found,
          "val_ran": val_ran,
          "val_ok": val_ok,
          "failed_line_count": failed_line_count,
          "solver_error": solver_error,
          "solver_stdout_chars": len(solver_stdout),
          "solver_stderr_chars": len(solver_stderr),
      }
  )

details_df = pd.DataFrame(rows).sort_values("problem").reset_index(drop=True)

summary_df = (
  details_df.groupby("final_status")
  .size()
  .rename("count")
  .reset_index()
  .sort_values(["count", "final_status"], ascending=[False, True])
)

n = len(details_df)
success_n = int((details_df["final_status"] == "success").sum()) if n else 0
success_rate = (success_n / n) if n else 0.0

print("Summary:")
print(summary_df.to_string(index=False))
print(f"\nTotal: {n}")
print(f"Success: {success_n}")
print(f"Success rate: {success_rate:.2%}")

print("\nDetails:")
print(details_df.to_string(index=False))

# expose canonical variable names for next persistence cell
SUMMARY_DF = summary_df
DETAILS_DF = details_df
SUMMARY_PROBLEMS_USED = SUMMARY_PROBLEMS

In [ ]:
# Cell 18: Persist summary/details to CSV (run only when you want to save)
from datetime import datetime

# Optional label to distinguish runs (edit as needed)
SAVE_LABEL = "qwen2_5_32b_instruct_kani_dnc_batch"

# Where to save metric CSVs
METRIC_DIR = OUTPUT_ROOT / "_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if "SUMMARY_DF" not in globals() or "DETAILS_DF" not in globals():
  raise RuntimeError("Run Cell 17 first to populate SUMMARY_DF and DETAILS_DF.")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
problems_tag = (
  f"{SUMMARY_PROBLEMS_USED[0]}_to_{SUMMARY_PROBLEMS_USED[-1]}"
  if len(SUMMARY_PROBLEMS_USED) > 1
  else SUMMARY_PROBLEMS_USED[0]
)

summary_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_summary_{ts}.csv"
details_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_details_{ts}.csv"

SUMMARY_DF.to_csv(summary_out, index=False)
DETAILS_DF.to_csv(details_out, index=False)

print("Saved:")
print("-", summary_out)
print("-", details_out)

# Parser for Problem Descriptions & Difference Checking

In [107]:
# Cell 17: Parser section setup (independent)
from pathlib import Path
import re
import json

# Re-declare all paths needed for parser/diff section (no dependency on earlier cells)
PROJECT_ROOT = Path.cwd().parents[1]  # run notebook from formalizer-scaling/src/pddl_gen
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_dnc" / "qwen2_5_32b_instruct_kani"

# Single-problem handle for parser + diff run
PARSER_PROBLEM = "p100"  # edit as needed

# Input files
NL_PROBLEM_PATH = DATA_DIR / "descriptions" / f"{PARSER_PROBLEM}_problem.txt"
LLM_PDDL_PATH = OUTPUT_ROOT / PARSER_PROBLEM / "cand_00" / "problem.pddl"

# Output folder/files
CAND_DIR = OUTPUT_ROOT / PARSER_PROBLEM / "cand_00"
CAND_DIR.mkdir(parents=True, exist_ok=True)

PARSED_PDDL_OUT = CAND_DIR / "parsed_problem.pddl"
DIFF_JSON_OUT = CAND_DIR / "parsed_vs_llm_diff.json"
DIFF_TXT_OUT = CAND_DIR / "parsed_vs_llm_diff.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem:", PARSER_PROBLEM)
print("NL exists:", NL_PROBLEM_PATH.exists())
print("LLM PDDL exists:", LLM_PDDL_PATH.exists())
print("Output dir:", CAND_DIR)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
Problem: p100
NL exists: True
LLM PDDL exists: True
Output dir: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_dnc/qwen2_5_32b_instruct_kani/p100/cand_00


In [17]:
# Cell 18: Helper 1 - normalize whitespace
def _norm_space(s: str) -> str:
  return re.sub(r"\s+", " ", s.strip())

In [18]:
# Cell 19: Helper 2 - split NL section into ordered statements
def _split_statements(section_text: str):
  text = section_text.replace(", and ", ", ")
  parts = [_norm_space(p) for p in text.split(",")]
  return [p for p in parts if p]

In [19]:
# Cell 20: Helper 3 - parse NL problem description into ordered facts
def parse_nl_problem_description(nl_text: str):
  text = _norm_space(nl_text)

  init_prefix = "As initial conditions I have that, "
  goal_prefix = "My goal is to have that "

  if init_prefix not in text or goal_prefix not in text:
      raise ValueError("Unexpected NL format: missing initial/goal prefixes.")

  init_start = text.index(init_prefix) + len(init_prefix)
  goal_start = text.index(goal_prefix)

  init_text = text[init_start:goal_start].strip()
  goal_text = text[goal_start + len(goal_prefix):].strip()

  init_statements = _split_statements(init_text)
  goal_statements = _split_statements(goal_text)

  init_facts = []
  goal_facts = []
  block_ids = set()

  re_clear = re.compile(r"^block (\d+) is clear\.$|^block (\d+) is clear$")
  re_on = re.compile(r"^block (\d+) is on top of block (\d+)\.$|^block (\d+) is on top of block (\d+)$")
  re_table = re.compile(r"^block (\d+) is on the table\.$|^block (\d+) is on the table$")
  re_hand = re.compile(r"^the hand is empty\.$|^the hand is empty$")

  def parse_stmt(stmt: str, in_goal: bool):
      st = stmt.strip()

      m = re_clear.match(st)
      if m:
          b = int(m.group(1) or m.group(2))
          block_ids.add(b)
          return ("clear", f"block{b}")

      m = re_on.match(st)
      if m:
          b1 = int(m.group(1) or m.group(3))
          b2 = int(m.group(2) or m.group(4))
          block_ids.update([b1, b2])
          return ("on", f"block{b1}", f"block{b2}")

      m = re_table.match(st)
      if m:
          b = int(m.group(1) or m.group(2))
          block_ids.add(b)
          return ("on-table", f"block{b}")

      m = re_hand.match(st)
      if m:
          if in_goal:
              raise ValueError("Unexpected 'the hand is empty' in goal section.")
          return ("arm-empty",)

      raise ValueError(f"Unrecognized statement: {st}")

  for s in init_statements:
      init_facts.append(parse_stmt(s, in_goal=False))

  for s in goal_statements:
      goal_facts.append(parse_stmt(s, in_goal=True))

  if not block_ids:
      raise ValueError("No block ids parsed from NL problem description.")

  n_blocks = max(block_ids)
  objects = [f"block{i}" for i in range(1, n_blocks + 1)]

  return {
      "objects": objects,
      "init_facts_ordered": init_facts,
      "goal_facts_ordered": goal_facts,
      "n_blocks": n_blocks,
  }

In [20]:
# Cell 21: Helper 4 - render parsed facts into problem PDDL text
def render_problem_pddl(problem_id: str, parsed):
  lines = []
  lines.append(f"(define (problem blocksworld-{problem_id})")
  lines.append("  (:domain blocksworld)")
  lines.append(f"  (:objects {' '.join(parsed['objects'])} )")
  lines.append("  (:init ")

  for fact in parsed["init_facts_ordered"]:
      if fact[0] == "clear":
          lines.append(f"    (clear {fact[1]})")
      elif fact[0] == "on":
          lines.append(f"    (on {fact[1]} {fact[2]})")
      elif fact[0] == "on-table":
          lines.append(f"    (on-table {fact[1]})")
      elif fact[0] == "arm-empty":
          lines.append("    (arm-empty)")
      else:
          raise ValueError(f"Unknown init fact: {fact}")

  lines.append("  )")
  lines.append("  (:goal (and ")

  for fact in parsed["goal_facts_ordered"]:
      if fact[0] == "clear":
          lines.append(f"    (clear {fact[1]})")
      elif fact[0] == "on":
          lines.append(f"    (on {fact[1]} {fact[2]})")
      elif fact[0] == "on-table":
          lines.append(f"    (on-table {fact[1]})")
      elif fact[0] == "arm-empty":
          lines.append("    (arm-empty)")
      else:
          raise ValueError(f"Unknown goal fact: {fact}")

  lines.append("  ))")
  lines.append(")")
  return "\n".join(lines) + "\n"

In [21]:
# Cell 22: Helper 5 - extract init/goal facts from a PDDL problem file (section-aware)
_fact_line_re = re.compile(r"\(\s*([a-zA-Z\-]+)([^()]*)\)")

def extract_section_facts_from_pddl(pddl_text: str):
  init_idx = pddl_text.find("(:init")
  goal_idx = pddl_text.find("(:goal")
  if init_idx == -1 or goal_idx == -1:
      raise ValueError("PDDL missing :init or :goal section.")

  init_text = pddl_text[init_idx:goal_idx]
  goal_text = pddl_text[goal_idx:]

  def _facts_from_section(sec: str):
      facts = []
      for m in _fact_line_re.finditer(sec):
          pred = m.group(1).strip()
          args = _norm_space(m.group(2)).split()

          if pred in {"clear", "on-table"} and len(args) == 1:
              facts.append((pred, args[0]))
          elif pred == "on" and len(args) == 2:
              facts.append((pred, args[0], args[1]))
          elif pred == "arm-empty" and len(args) == 0:
              facts.append((pred,))
      return facts

  return {
      "init_facts": _facts_from_section(init_text),
      "goal_facts": _facts_from_section(goal_text),
  }

In [22]:
# Cell 23: Helper 6 - section-aware, order-insensitive diff
def diff_fact_sections(parsed_pddl_text: str, llm_pddl_text: str):
  parsed_secs = extract_section_facts_from_pddl(parsed_pddl_text)
  llm_secs = extract_section_facts_from_pddl(llm_pddl_text)

  parsed_init = set(parsed_secs["init_facts"])
  parsed_goal = set(parsed_secs["goal_facts"])
  llm_init = set(llm_secs["init_facts"])
  llm_goal = set(llm_secs["goal_facts"])

  return {
      "init_missing_in_llm": sorted(parsed_init - llm_init),
      "init_extra_in_llm": sorted(llm_init - parsed_init),
      "goal_missing_in_llm": sorted(parsed_goal - llm_goal),
      "goal_extra_in_llm": sorted(llm_goal - parsed_goal),
      "init_match_count": len(parsed_init & llm_init),
      "goal_match_count": len(parsed_goal & llm_goal),
      "init_total_parsed": len(parsed_init),
      "goal_total_parsed": len(parsed_goal),
      "init_total_llm": len(llm_init),
      "goal_total_llm": len(llm_goal),
      "perfect_match": (
          parsed_init == llm_init and
          parsed_goal == llm_goal
      ),
  }

In [23]:
# Cell 24: Helper 7 - concise mismatch-only diff report text
def build_diff_report_text(problem_id: str, diff_result: dict):
  lines = []
  lines.append(f"Problem: {problem_id}")
  lines.append(f"Perfect match: {diff_result['perfect_match']}")
  lines.append("")

  lines.append(f"Init missing in LLM ({len(diff_result['init_missing_in_llm'])}):")
  for fact in diff_result["init_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Init extra in LLM ({len(diff_result['init_extra_in_llm'])}):")
  for fact in diff_result["init_extra_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal missing in LLM ({len(diff_result['goal_missing_in_llm'])}):")
  for fact in diff_result["goal_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal extra in LLM ({len(diff_result['goal_extra_in_llm'])}):")
  for fact in diff_result["goal_extra_in_llm"]:
      lines.append(f"- {fact}")

  return "\n".join(lines) + "\n"

In [108]:
# Cell 25: Parser + diff run (single problem handle, persists outputs to pXXX/cand_00)
if not NL_PROBLEM_PATH.exists():
  raise FileNotFoundError(f"Missing NL description: {NL_PROBLEM_PATH}")
if not LLM_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing LLM-generated PDDL: {LLM_PDDL_PATH}")

nl_text = NL_PROBLEM_PATH.read_text()
llm_pddl_text = LLM_PDDL_PATH.read_text()

parsed_struct = parse_nl_problem_description(nl_text)
parsed_pddl_text = render_problem_pddl(PARSER_PROBLEM, parsed_struct)
diff_result = diff_fact_sections(parsed_pddl_text, llm_pddl_text)
diff_text = build_diff_report_text(PARSER_PROBLEM, diff_result)

PARSED_PDDL_OUT.write_text(parsed_pddl_text)
DIFF_JSON_OUT.write_text(json.dumps(diff_result, indent=2))
DIFF_TXT_OUT.write_text(diff_text)

print("Saved:")
print("-", PARSED_PDDL_OUT)
print("-", DIFF_JSON_OUT)
print("-", DIFF_TXT_OUT)
print("")
print("Mismatch summary:")
print(f"- init missing: {len(diff_result['init_missing_in_llm'])}")
print(f"- init extra:   {len(diff_result['init_extra_in_llm'])}")
print(f"- goal missing: {len(diff_result['goal_missing_in_llm'])}")
print(f"- goal extra:   {len(diff_result['goal_extra_in_llm'])}")
print(f"- perfect_match: {diff_result['perfect_match']}")

Saved:
- /home/sagemaker-user/formalizer-scaling/output/direct_pddl_dnc/qwen2_5_32b_instruct_kani/p100/cand_00/parsed_problem.pddl
- /home/sagemaker-user/formalizer-scaling/output/direct_pddl_dnc/qwen2_5_32b_instruct_kani/p100/cand_00/parsed_vs_llm_diff.json
- /home/sagemaker-user/formalizer-scaling/output/direct_pddl_dnc/qwen2_5_32b_instruct_kani/p100/cand_00/parsed_vs_llm_diff.txt

Mismatch summary:
- init missing: 0
- init extra:   0
- goal missing: 0
- goal extra:   0
- perfect_match: True
